# Banking Credit Risk — German Credit (Retail-Banking Domain)

End-to-end **credit scoring** for a bank using UCI German Credit data and ANN.
Same pipeline as loan default with emphasis on **approve/deny** policy and fair lending checks.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Banks must estimate **probability of default (PD)** for pricing and capital requirements (Basel). False negatives (approving bad loans) are costly.


In [ ]:
# UCI German Credit (statlog)
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
names = [f"A{i}" for i in range(1, 21)] + ["class"]
df = pd.read_csv(URL, sep=r"\s+", header=None, names=names)
# class: 1 = good, 2 = bad -> default = bad
df["default"] = (df["class"] == 2).astype(int)
df = df.drop(columns=["class"])
print(df.head())


In [ ]:
# EDA — all features categorical in original encoding
print(df["default"].value_counts(normalize=True))
sns.countplot(data=df, x="default")
plt.title("Default rate (1=bad credit)")
plt.show()


In [ ]:
# Encode categoricals as integers then one-hot
X_df = pd.get_dummies(df.drop(columns=["default"]), drop_first=True)
y = df["default"].values
X = X_df.values.astype(np.float32)


In [ ]:
# Train / validation / test split (stratified for classification)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)  # ~70/15/15

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Val:", X_val_s.shape, "Test:", X_test_s.shape)


In [ ]:
model = models.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", "AUC"])
model.summary()


In [ ]:
checkpoint = callbacks.ModelCheckpoint(
    "ann_loan_best.keras", monitor="val_loss", save_best_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
cb_list = [checkpoint, early_stop, reduce_lr]


In [ ]:
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=80,
    batch_size=32,
    callbacks=cb_list,
    verbose=1,
)

# Loss curves
pd.DataFrame(history.history).plot(figsize=(10, 4))
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.show()

# Test metrics
y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
try:
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
except Exception:
    pass
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Test)")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()


In [ ]:
loaded = keras.models.load_model("ann_loan_best.keras")
for i in range(min(5, len(X_test_s))):
    p = loaded.predict(X_test_s[i:i+1], verbose=0)[0, 0]
    print(f"Applicant {i}: P(default)={p:.3f}, pred={int(p>=0.5)}, actual={y_test[i]}")
import joblib
joblib.dump(scaler, "loan_scaler.pkl")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
